In [0]:
# Parameters (can be overridden from the Job)
dbutils.widgets.text("schema", "gold", "Schema to validate")
dbutils.widgets.text("fail_on_issues", "true", "Fail the job on issues (true/false)")
dbutils.widgets.text("expected_rows", "180519", "Expected fact rows (0 = skip)")

schema = dbutils.widgets.get("schema")
fail_on_issues = dbutils.widgets.get("fail_on_issues").lower() == "true"
expected_rows = int(dbutils.widgets.get("expected_rows") or 0)
fact = f"{schema}.fct_order_items"

In [0]:
def orphan(dim, key):
    return (f"SELECT COUNT(*) FROM {fact} f LEFT JOIN {schema}.{dim} d "
            f"ON f.{key} = d.{key} WHERE d.{key} IS NULL")

def not_unique(dim, key):
    return f"SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM {schema}.{dim}"

checks = {
    "Duplicate order_item_key (grain)": f"SELECT COUNT(*) - COUNT(DISTINCT order_item_key) FROM {fact}",
    "Null order_item_key": f"SELECT COUNT(*) FROM {fact} WHERE order_item_key IS NULL",
    "Negative sales": f"SELECT COUNT(*) FROM {fact} WHERE sales < 0",
    "discount_rate outside 0-1": f"SELECT COUNT(*) FROM {fact} WHERE discount_rate < 0 OR discount_rate > 1",
    "Fact key without matching geography": orphan("dim_geography", "geography_key"),
    "Fact key without matching product": orphan("dim_product", "product_key"),
    "Fact key without matching shipping mode": orphan("dim_shipping_mode", "shipping_mode_key"),
    "Fact key without matching date": orphan("dim_date", "date_key"),
    "Non-unique key in dim_geography": not_unique("dim_geography", "geography_key"),
    "Non-unique key in dim_product": not_unique("dim_product", "product_key"),
    "Non-unique key in dim_shipping_mode": not_unique("dim_shipping_mode", "shipping_mode_key"),
    "Non-unique key in dim_date": not_unique("dim_date", "date_key"),
}

In [0]:
rows = []
for name, sql in checks.items():
    n = int(spark.sql(sql).first()[0])
    rows.append((name, n, "OK" if n == 0 else "REVIEW"))

if expected_rows:
    actual = spark.table(fact).count()
    diff = abs(actual - expected_rows)
    rows.append((f"Fact row count equals expected ({expected_rows:,})", diff,
                 "OK" if diff == 0 else "REVIEW"))

result = spark.createDataFrame(rows, ["check", "issues", "status"])
display(result)
failed = [r for r in rows if r[1] != 0]

check,issues,status
Duplicate order_item_key (grain),0,OK
Null order_item_key,0,OK
Negative sales,0,OK
discount_rate outside 0-1,0,OK
Fact key without matching geography,0,OK
Fact key without matching product,0,OK
Fact key without matching shipping mode,0,OK
Fact key without matching date,0,OK
Non-unique key in dim_geography,0,OK
Non-unique key in dim_product,0,OK


In [0]:
if failed and fail_on_issues:
    detail = "; ".join(f"{name} ({n})" for name, n, _ in failed)
    raise Exception(f"{len(failed)} data quality check(s) failed: {detail}")

print(f"All {len(rows)} data quality checks passed")

All 13 data quality checks passed
